# Cardinal 101

An application of Cardinal for event analysis that is a good starting point for new users

This notebook contains detailed explanations of algorithms. For a clean version without details, see ```cardinal101_clean.ipynb```

This notebook demonstrates event analysis—a common workflow using known signal location and origin time to find signals of interest.

**Event:** Bolide recorded at the I57US array  
**Date:** 2016-06-02  

By the Cardinal Team.<br>
*Portions of this code were developed with the assistance of a large language model (LLM). All content has been reviewed and validated by the authors.*

In [ ]:
%%time
# Uncomment line below for interactive plots in JupyterLab.
# %matplotlib widget
from obspy.clients.fdsn import Client as FDSNClient
from obspy.core import AttribDict
from obspy import UTCDateTime
from obspy.geodetics import gps2dist_azimuth
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import cardinal
import cardinal_fk
from dask.distributed import Client as DaskClient

In [ ]:
def get_stream_with_coords(network, station, channel, t1, t2, location='??', provider='IRIS'):
    """
    Get ObsPy Stream with SAC coordinate headers attached.
    
    Parameters
    ----------
    network : str
        Network code (e.g., 'IM')
    station : str
        Station code(s), wildcards allowed (e.g., 'I51*')
    channel : str
        Channel code(s), wildcards allowed (e.g., 'BDF')
    t1 : UTCDateTime
        Start time
    t2 : UTCDateTime
        End time
    location : str, optional
        Location code(s), default '??'
    provider : str, optional
        FDSN provider, default 'IRIS'
    
    Returns
    -------
    obspy.Stream
        Stream with tr.stats.sac.{stla, stlo, stel} attached
    """
    client = FDSNClient(provider)
    
    # Get station metadata
    inv = client.get_stations(network=network, station=station, channel=channel,
                              starttime=t1, endtime=t2)
    
    # Build station list for waveform request
    stations_wf = []
    station_coords = {}
    for net in inv:
        for sta in net:
            stations_wf.append(sta.code)
            station_coords[sta.code] = {
                'lat': sta.latitude,
                'lon': sta.longitude,
                'elv': sta.elevation
            }
    
    station_str = ','.join(stations_wf)
    
    # Get waveforms
    st = client.get_waveforms(network, station_str, location, channel, t1, t2, 
                              attach_response=True)
    
    # Attach SAC headers
    for tr in st:
        coords = station_coords[tr.stats.station]
        tr.stats.sac = AttribDict({
            "stla": coords['lat'],
            "stlo": coords['lon'],
            "stel": coords['elv']
        })
    
    return st

## 1 Load and Inspect data

First, read data from ObsPy using a user-defined array and time window. Then inspect it: visualize raw data, remove bad traces, and assess spectral content.

### a) Load Data

In [ ]:
# Event parameters
evla = 33.8
evlo = -110.9
etime = UTCDateTime('2016-06-02T10:56:32')

# Data request
t1 = UTCDateTime("2016-06-02T11:15:00")
t2 = UTCDateTime("2016-06-02T11:45:00")
st = get_stream_with_coords('IM', 'I57*', 'BDF', t1, t2, location='--,??')

# Calculate great circle backazimuth
distance_m, azimuth, back_azimuth = gps2dist_azimuth(
    st[0].stats.sac.stla, st[0].stats.sac.stlo,
    evla, evlo
)
distance_km = distance_m / 1000.0
backazimuth = azimuth

print(f"Distance: {distance_km:.2f} km")
print(f"Backazimuth: {backazimuth:.2f}°")

### b) View data and remove any bad traces

In [ ]:
fig = st.plot(handle=True, method='full', type='relative', equal_scale=False, size=(900, 500))

In [ ]:
st_corr = cardinal.plot_data_quality(st, amp_units='Pressure [Pa]', reverse_polarities=True, return_stream=True)

In [ ]:
fig = st_corr.plot(handle=True, method='full', type='relative', equal_scale=True, size=(900, 500))

### c) Spectral Analysis
Cardinal provides two helper functions for plotting spectrograms and scalograms.

In [ ]:
cardinal.plot_spectrogram(
    st_corr, element='I57H1', bandpass=[0.1,10], nperseg=2**8, log_scale=True,
    amp_units='Pressure [Pa]', normalize=True, title='I57US',
    )

In [ ]:
cardinal.plot_scalogram(
    st_corr, element='I57H1', bandpass=[0.1,10], trim_stream=[200,1200],
    normalize=True, amp_units='Pressure [Pa]', title='I57US'
    )

## 2. Array Geometry and Coordinates
Before array processing, get familiar with the array by plotting its coordinates.

In [ ]:
# Extract array coordinates
x_km, y_km, lat0, lon0 = cardinal_fk.get_array_coordinates(st_corr, return_centroid=True)

print(f"Array centroid: lat={lat0:.4f}°, lon={lon0:.4f}°")
print(f"Array aperture: Δx={x_km.max()-x_km.min():.3f} km, Δy={y_km.max()-y_km.min():.3f} km, Δ={np.sqrt((x_km.max()-x_km.min())+ (y_km.max()-y_km.min())):.3f} km")
print(f"Number of sensors: {len(x_km)}")

# Plot array geometry
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x_km, y_km, s=30, color='steelblue', edgecolor='black', linewidths=0.5, zorder=5)
ax.scatter(0, 0, s=150, color='red', marker='+', zorder=6, linewidths=2.5, label='centroid')
ax.set_xlabel('x  [km]  (East)', fontsize=11)
ax.set_ylabel('y  [km]  (North)', fontsize=11)
ax.set_title(f'I57 Array Geometry ({len(x_km)} sensors)', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Array Response Function (ARF)
Spectral analysis shows signals are strongest in the 0.1–10 Hz band. Before array processing, evaluate the array response at low, high, and broadband frequencies.

The array response function (ARF) describes how well the array resolves plane waves across slowness directions, revealing the beam pattern and sidelobes that can affect FK analysis.

### Method

An array records a wavefield at $N$ sensors. A plane wave arriving with **trace velocity** $v_{app}$ from **backazimuth** $\theta$ reaches each sensor at a slightly different time. The 2D **slowness vector** $\mathbf{u} = (u_x, u_y)$ describes this plane wave:

$$u_x = \frac{\sin\theta}{v_{app}}, \quad u_y = \frac{\cos\theta}{v_{app}}$$

where $u_x$ is the East component (E–W slowness) and $u_y$ is the North component (N–S slowness), in units of s/km.

For sensor $i$ located at position $\mathbf{r}_i = (x_i, y_i)$ relative to the array centroid, the time delay of the plane wave is:

$$\tau_i = \mathbf{r}_i \cdot \mathbf{u} = u_x \cdot x_i + u_y \cdot y_i$$

In the frequency domain, a time shift $\tau_i$ becomes a phase shift $e^{-i 2\pi f \tau_i}$ (Fourier Shift Theorem).

For a plane wave with slowness vector $\mathbf{u} = (u_x, u_y)$ at frequency $f$, the ARF is:

$$A(u_x, u_y) = \left| \frac{1}{N} \sum_{i=1}^N e^{-i 2 \pi f \mathbf{r}_i \cdot \mathbf{u}} \right|^2$$

In [ ]:
# Compute and plot ARF (broadband only by default)
#fig, ax = cardinal_fk.plot_arf(
#   x_km, y_km,
#    freq_low=0.1,
#    freq_high=10.0,
#    smax=3.6,
#    ngrid=200
#)
#plt.show()

# To see all three frequency panels, use: show_all_frequencies=True
fig, axes = cardinal_fk.plot_arf(x_km, y_km, freq_low=0.1, freq_high=10.0,
                                  smax=3.6, ngrid=200, show_all_frequencies=True)
plt.show()

## 4. Sliding-Window F-K Analysis
Apply F-K analysis in a sliding window over the frequency band containing the signal. This identifies time segments of coherent signal, and the corresponding backazimuth and trace velocity of the best fitting plane wave.

The beamformed signal for slowness $(u_x, u_y)$ at frequency $f$ is:

$$B(f, u_x, u_y) = \frac{1}{N} \sum_{i=1}^{N} U_i(f) \cdot e^{+i 2\pi f (u_x x_i + u_y y_i)}$$

where $U_i(f)$ is the Fourier transform of sensor $i$'s waveform.

The f-k power integrated over a frequency band $[f_1, f_2]$ is:

$$P(u_x, u_y) = \int_{f_1}^{f_2} \left| B(f, u_x, u_y) \right|^2 df$$

The slowness that maximises $P(u_x, u_y)$ is the estimated slowness of the incoming wave.

Semblance $S \in [0, 1]$ normalises the beam power by the total power across all sensors, making it independent of signal amplitude:

$$S(u_x, u_y) = \frac{\int_{f_1}^{f_2} \left| B(f, u_x, u_y) \right|^2 df}{\frac{1}{N}\int_{f_1}^{f_2} \sum_{i=1}^{N} \left| U_i(f) \right|^2 df}$$

A semblance of 1.0 means perfect coherence; 0 means no coherence.


In [ ]:
# F-K parameters
FK_FREQ_MIN = 0.1  # Hz
FK_FREQ_MAX = 10.0  # Hz
FK_SMAX = 3.6      # s/km
FK_NGRID = 101

# Sliding window parameters
FK_WINDOW_LENGTH = 20.0    # seconds
FK_OVERLAP_PERCENT = 50.0  # percent

# Prepare data for sliding window (use full time range)
fs = st_corr[0].stats.sampling_rate
data_all = np.array([tr.data.astype(float) for tr in st_corr])

print(f"Running sliding window F-K analysis...")
print(f"  Window: {FK_WINDOW_LENGTH}s, Overlap: {FK_OVERLAP_PERCENT}%")
print(f"  Note: First run may be slower due to JIT compilation")

# Run sliding window F-K
T_fk, B_fk, V_fk, S_fk = cardinal_fk.sliding_window_fk(
    data_all, x_km, y_km, fs,
    fmin=FK_FREQ_MIN,
    fmax=FK_FREQ_MAX,
    window_length=FK_WINDOW_LENGTH,
    overlap_percent=FK_OVERLAP_PERCENT,
    smax=FK_SMAX,
    ngrid=FK_NGRID
)

print(f"\n✓ Complete!")

In [ ]:
# Use cardinal's plotting function (from cardinal101.ipynb)
def plot_sliding_window_custom(st, element, T, B, V, C=None, v_min=0, v_max=5., 
                        semblance_threshold=None, twin_plot=None, clim=[0,1], figsize=(10,6),
                        baz_line=None, ylim_baz=None):

    tr = st.select(station=element)[0]

    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(
        3, 2,
        width_ratios=[40, 1.0],
        height_ratios=[1, 1, 1],
        wspace=0.02,
        hspace=0.20
    )

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
    ax3 = fig.add_subplot(gs[2, 0], sharex=ax1)
    cax = fig.add_subplot(gs[1:, 1])  # colorbar only beside bottom two panels

    t_tr = np.arange(0, tr.stats.npts * tr.stats.delta, tr.stats.delta)
    ax1.plot(t_tr, tr.data / np.max(np.abs(tr.data)), 'k-')
    ax1.tick_params(labelbottom=False)
    ax1.set_ylabel('Normalized\nAmplitude')

    if C is not None:
        if semblance_threshold is not None:
            ix2 = np.where(C < semblance_threshold)
            ax2.scatter(T[ix2], B[ix2], s=0.05, c='lightgray')
            ix = np.where(C >= semblance_threshold)
            sc2 = ax2.scatter(T[ix], B[ix], s=4, c=C[ix], vmin=clim[0], vmax=clim[1], cmap='hot_r')
        else:
            sc2 = ax2.scatter(T, B, s=4, c=C, vmin=clim[0], vmax=clim[1], cmap='hot_r')
    else:
        ax2.plot(T, B, 'k.')
        sc2 = None

    if baz_line is not None:
        ax2.axhline(y=baz_line, color='blue', linestyle='--', linewidth=2, label=f'GC BAZ: {baz_line:.1f}°')
        ax2.legend(loc='upper right', fontsize=8)

    ax2.set_ylim(ylim_baz if ylim_baz is not None else [0, 360])
    ax2.set_ylabel('Backazimuth')
    if twin_plot is not None:
        ax2.set_xlim(twin_plot)
    ax2.tick_params(labelbottom=False)

    if C is not None:
        if semblance_threshold is not None:
            ix2 = np.where(C < semblance_threshold)
            ax3.scatter(T[ix2], V[ix2], s=0.05, c='lightgray')
            ix = np.where(C >= semblance_threshold)
            sc3 = ax3.scatter(T[ix], V[ix], s=4, c=C[ix], vmin=clim[0], vmax=clim[1], cmap='hot_r')
        else:
            sc3 = ax3.scatter(T, V, s=4, c=C, vmin=clim[0], vmax=clim[1], cmap='hot_r')
    else:
        ax3.plot(T, V, 'k.')

    ax3.set_ylim([v_min, v_max])
    ax3.set_ylabel('Phase vel.')
    ax3.set_xlabel('Time [s] after ' + str(tr.stats.starttime).split('.')[0].replace('T', ' '))
    ax3.set_xlim([t_tr[0], t_tr[-1]])

    if C is not None and sc2 is not None:
        fig.colorbar(sc2, cax=cax, label='Semblance')
        
    plt.tight_layout()
    
    return fig, (ax1, ax2, ax3)

# Plot
st_f = st_corr.copy()
st_f.taper(type='cosine', max_percentage=0.01)
st_f.filter('bandpass', freqmin=FK_FREQ_MIN, freqmax=FK_FREQ_MAX)

plot_sliding_window_custom(st_f, 'I57H1', T_fk, B_fk, V_fk, C=S_fk,
                          v_min=0.3, v_max=0.5, clim=[0.2, 1],
                          baz_line=backazimuth, ylim_baz=[75, 90])
plt.show()

## 5. Single-Window F-K Analysis
To understand the resolution of the best-fitting slowness vector, compute and view an F-K plot over a specified time window

In [ ]:
# F-K parameters
FK_FREQ_MIN = 0.1  # Hz
FK_FREQ_MAX = 10.0  # Hz
FK_SMAX = 3.6      # s/km
FK_NGRID = 401

# Time window
FK_TIME_START = 726  # seconds from start
FK_TIME_END = 766

# Extract and prepare data
time_start_fk = st_corr[0].stats.starttime + FK_TIME_START
time_end_fk = st_corr[0].stats.starttime + FK_TIME_END
st_fk = st_corr.copy()
st_fk.trim(starttime=time_start_fk, endtime=time_end_fk)
st_fk.detrend('linear')
st_fk.taper(type='cosine', max_percentage=0.05)

# Get data array
fs = st_fk[0].stats.sampling_rate
min_length = min([tr.stats.npts for tr in st_fk])
data_fk = np.array([tr.data[:min_length].astype(float) for tr in st_fk])

print(f"Running F-K analysis on {FK_TIME_END-FK_TIME_START}s window...")

# F-K analysis
sx_vec, sy_vec, power_grid, semblance_grid = cardinal_fk.fk_analysis(
    data_fk, x_km, y_km, fs,
    FK_FREQ_MIN, FK_FREQ_MAX,
    smax=FK_SMAX, ngrid=FK_NGRID
)

# Plot results
fig, axes, results = cardinal_fk.plot_fk(
    sx_vec, sy_vec, power_grid, semblance_grid,
    FK_FREQ_MIN, FK_FREQ_MAX,
    power_vmin_db=-8
)
plt.show()

# 6. Multifrequency Analysis

Standard array processing applies FK analysis across a single, fixed frequency band. This works well when a signal spans the full band with adequate SNR, but many signals may only span a short bandwidth and the SNR typically varies strongly with frequency. Multifrequency analysis addresses this by partitioning the data into discrete frequency–time blocks and processing each independently, which improves detection sensitivity and allows the coherence structure of the wavefield to be characterized as a function of frequency.

The approach is described in:

> Ronac Giannone, M., Arrowsmith, S. J., & Silber, E. A. (2026). Cardinal: Seismic and geoacoustic array processing. *Seismological Research Letters*, 97(1), 487–500.

The workflow consists of four stages:

1. **Segmentor** — Partitions the data into processing blocks defined by a set of frequency bands and associated time windows.
2. **Adaptive Array** — Selects an appropriate array configuration (e.g., subarray geometry) for each frequency band, accounting for aperture constraints.
3. **Array Processor** — Applies sliding-window FK analysis within each frequency band using the algorithms described in Section 4.
4. **Aggregator** — Combines per-block FK results to form detections in the joint time–frequency domain.

**Key trade-off.** Narrower frequency bands improve frequency resolution but reduce FK resolution (i.e., slowness precision). Choose band widths and time windows with this in mind.

> ⚠️ This section currently uses legacy ObsPy-based FK processing rather than Cardinal's native fk algorithms.

---

### a) Segmentor: Define frequency bands and time windows

The first step is to specify a set of frequency bands that together span the broadband range of interest. Each band should be paired with a time window length appropriate for that frequency: longer windows at low frequencies (to capture enough cycles), shorter windows at high frequencies.

In [ ]:
# To use third-octave bands:
f_bands = cardinal.make_custom_fbands(f_min=0.1, f_max=11, win_min=30, win_max=90., type='third_octave')
f_bands['fmax'].values[-1] = np.round(f_bands['fmax'].values[-1],0) # rounding so f_max becomes Nyquist

# To use PMCC frequency bands instead, use:
#f_bands = cardinal.pmcc_fbands()
#f_bands = cardinal.extend_pmcc_fbands(f_bands, st[0].stats.sampling_rate/2)

f_bands

### b) Use Adaptive Array to identify subarrays per band
Skip this section if not using a multi-aperture array

In [ ]:
_, _, k = cardinal.clusters(st_corr, plot=True)

In [ ]:
_, subarrays_stnms = cardinal.adaptive_array(st_corr, f_bands, array_type='infrasound', plot=True, n_clusters=k)

In [ ]:
st_subarrays = cardinal.retrieve_subarray_data(st_corr, subarrays_stnms)

### c) Applying Sliding-Window FK in each band

In [ ]:
%%time
client = DaskClient(processes=True, threads_per_worker=1, n_workers=6, memory_limit='3GB')
T_adaptive, B_adaptive, V_adaptive, S_adaptive = cardinal.sliding_time_array_fk_multifreq(
    st_subarrays, f_bands, client, signal_type='infrasound', adaptive_array=True
    )
client.close()

# Without the Adaptive Array, use:
#client = DaskClient(processes=True, threads_per_worker=1, n_workers=6, memory_limit='3GB')
#T, B, V, S = cardinal.sliding_time_array_fk_multifreq(
#    st_corr, f_bands, client, signal_type='infrasound', adaptive_array=False
#    )
#client.close()

In [ ]:
cardinal.plot_sliding_window_multifreq(
    st_corr, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, semblance_threshold=0.5,
    clim_vtr=[0.2,0.45], clim_baz=[-90,90], normalize=True,
    bandpass=[0.5,5], GT_baz=backazimuth, delay_times=None, amp_units='Pressure [Pa]', 
    title='I57US' + ' - Adaptive Array\n' + str(distance_km) + ' [km]')

# Without the Adaptive Array, use:
#cardinal.plot_sliding_window_multifreq(
#    st_corr, f_bands, T, B, V, S, semblance_threshold=0.5,
#    clim_vtr=[0.2,0.45], clim_baz=[-90,90], normalize=True,
#    bandpass=[0.5,5], GT_baz=backazimuth, delay_times=None, amp_units='Pressure [Pa]', 
#    title='I57US\n' + str(distance_km) + ' [km]')

### d) Use Aggregator to identify unique detections

In [ ]:
ref_time = st[0].stats.starttime.matplotlib_date
ix, pixels_in_families, families = cardinal.make_families(
    T_adaptive, B_adaptive, V_adaptive, S_adaptive, f_bands, ref_time,
    dist_threshold=1, min_pixels=100, sigma_t=2, sigma_f=2, sigma_b=10, p_threshold=0.075,
    family_grouping='adaptive_kdtree_window'
    )
cardinal.df_families(ref_time, families)

# Without the Adaptive Array, use:
#ref_time = st[0].stats.starttime.matplotlib_date
#ix, pixels_in_families, families = cardinal.make_families(
#    T, B, V, S, f_bands, ref_time,
#    dist_threshold=1, min_pixels=100, sigma_t=2, sigma_f=2, sigma_b=10, p_threshold=0.075,
#    family_grouping='adaptive_kdtree_window'
#    )
#cardinal.df_families(ref_time, families)

In [ ]:
cardinal.plot_sliding_window_multifreq(st_corr, f_bands, T_adaptive, B_adaptive, V_adaptive, S_adaptive, 
                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], bandpass=[0.5,5], normalize=True,
                                       GT_baz=backazimuth, amp_units='Pressure [Pa]',
                                       pixels_in_families=pixels_in_families, ix=ix, 
                                       title='I57US' + ' - Adaptive Array - Aggregator\n' + str(distance_km) + ' [km]')

# Without the Adaptive Array, use:
#cardinal.plot_sliding_window_multifreq(st_corr, f_bands, T, B, V, S, 
#                                       clim_vtr=[0.2,0.45], clim_baz=[-90,90], bandpass=[0.5,5], normalize=True,
#                                       GT_baz=backazimuth, amp_units='Pressure [Pa]',
#                                       pixels_in_families=pixels_in_families, ix=ix, 
#                                       title='I57US' + ' - Adaptive Array - Aggregator\n' + str(distance_km) + ' [km]')